# 🐪 Aero-CamelBERT-X — Phase 3 (v3)

**Tri-Method XAI with content-word filtering.**

**v3 changes (on top of v2):**
- ✅ New `is_content_word()` utility — filters out Arabic stopwords,
  prepositions, conjunctions, single-letter pieces, punctuation, digits,
  and special tokens
- ✅ **Negation particles kept** (لا، ليس، لم، ما) — they carry stance
- ✅ Bar charts now show top-15 *content* words instead of `و`, `في`, `[SEP]`
- ✅ Heatmaps now show content tokens only (top-40 by absolute score)
- ✅ Global top-50 aggregation also content-only
- ✅ **Raw attribution CSVs untouched** — filtering is display-time only,
  so reviewers asking about stopword behavior can re-analyze later

**v2 → v3: what stays the same**
- Arabic reshape + bidi pipeline for correct RTL rendering
- Adaptive SHAP `max_evals = max(400, 2 × n_tokens)`
- Resumable CSVs

**If you already ran v2 successfully:** you do **not** need to re-run cells
6, 7, 8 (attention/LIME/SHAP). Just run cells 1–5, then jump to cell 9
onward — the new figures will be generated from your existing CSVs.

**Inputs.** Phase 2 checkpoint + existing v2 attribution CSVs (if any).
**Outputs.** New figures using filtered tokens, plus content-filtered
global-top-50 CSV.


## 1. Mount Drive + reload config

In [ ]:
from google.colab import drive
import json, os

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/aero_camelbert_x'
with open(f'{PROJECT_DIR}/configs/config.json') as f:
    CONFIG = json.load(f)

XAI_DIR = f'{CONFIG["log_dir"]}/xai'
FIG_DIR = f'{XAI_DIR}/figures'
os.makedirs(XAI_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
print('XAI dir:', XAI_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
XAI dir: /content/drive/MyDrive/aero_camelbert_x/logs/xai


## 2. Install

In [ ]:
!pip install -q --upgrade \
    "lime>=0.2.0" \
    "shap>=0.47" \
    "arabic-reshaper>=3.0" \
    "python-bidi>=0.4.2"
!apt-get install -y -q fonts-noto-core fonts-noto fonts-dejavu 2>&1 | tail -2
print('Install complete.')

fonts-noto is already the newest version (20201225-1build1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
Install complete.


## 3. Imports + Arabic-rendering helpers

In [ ]:
import os, json, time, re
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager

import arabic_reshaper
from bidi.algorithm import get_display

def find_arabic_font():
    candidates = [
        '/usr/share/fonts/truetype/noto/NotoNaskhArabic-Regular.ttf',
        '/usr/share/fonts/truetype/noto/NotoSansArabic-Regular.ttf',
        '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    for f in font_manager.findSystemFonts():
        name = os.path.basename(f).lower()
        if 'arabic' in name or 'naskh' in name:
            return f
    return None

ARABIC_FONT = find_arabic_font()
AR_FONT_PROP = None
if ARABIC_FONT:
    font_manager.fontManager.addfont(ARABIC_FONT)
    prop = font_manager.FontProperties(fname=ARABIC_FONT)
    print(f'Arabic font: {ARABIC_FONT}')

AR_RE = re.compile(r'[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\uFB50-\uFDFF\uFE70-\uFEFF]')

def contains_arabic(s):
    return bool(AR_RE.search(s))

def strip_wp_prefix(tok):
    return tok[2:] if tok.startswith('##') else tok

def reshape_ar(tok):
    if not isinstance(tok, str):
        return str(tok)
    core = strip_wp_prefix(tok)
    if contains_arabic(core):
        try:
            return get_display(arabic_reshaper.reshape(core))
        except Exception:
            return core
    return tok

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Arabic font: /usr/share/fonts/truetype/noto/NotoNaskhArabic-Regular.ttf
Device: cuda


## 4. Content-word filter

The core of v3. We define `is_content_word(token)` which returns `False`
for tokens that should be excluded from the explanations:

- **Special tokens** — `[CLS]`, `[SEP]`, `[PAD]`, `[UNK]`, `[MASK]`
- **Punctuation** — `. , ! ? " ' ( ) : ; — …` and Arabic equivalents
- **Digits** — Arabic-Indic and ASCII numerals
- **Single Arabic letters** — usually broken WordPiece fragments
- **Arabic stopwords** — particles, prepositions, conjunctions, pronouns
- **WordPiece pieces ≤ 2 chars** when their core strips to a stopword

**Kept (negation particles):** لا، ليس، لم، ما، لن، غير، بدون — these carry
semantic stance per your spec.

In [ ]:
# Special transformer tokens (always filter)
SPECIAL_TOKENS = {'[CLS]', '[SEP]', '[PAD]', '[UNK]', '[MASK]', '<s>', '</s>'}

# Punctuation: ASCII + Arabic + general
PUNCT = set('.,!?؟،؛:;"\'()[]{}<>/-—–…«»“”‘’…·•|')
# Add Arabic-specific punctuation
PUNCT |= {'\u060C', '\u061B', '\u061F', '\u066D'}  # ، ؛ ؟ ٭

# Digit characters
DIGITS = set('0123456789٠١٢٣٤٥٦٧٨٩')

# Negation particles to KEEP (per user requirement)
KEEP_NEGATION = {'لا', 'ليس', 'لم', 'ما', 'لن', 'غير', 'بدون', 'ليست'}

# Arabic stopwords — combined from CAMeL Tools, NLTK Arabic, and the standard
# Khoja stoplist. Coverage of ~250 most frequent function words.
ARABIC_STOPWORDS = {
    # Conjunctions
    'و', 'ف', 'ثم', 'أو', 'أم', 'لكن', 'لكنّ', 'بل', 'حتى',
    # Prepositions
    'في', 'من', 'إلى', 'على', 'عن', 'ب', 'ل', 'ك', 'مع',
    'عند', 'لدى', 'بعد', 'قبل', 'فوق', 'تحت', 'بين', 'أمام', 'وراء',
    'حول', 'ضد', 'دون', 'سوى', 'خلال', 'وفق', 'حسب',
    # Demonstratives
    'هذا', 'هذه', 'ذلك', 'تلك', 'هؤلاء', 'أولئك', 'هنا', 'هناك',
    'هاتان', 'هذان', 'ذاك',
    # Relative pronouns
    'الذي', 'التي', 'الذين', 'اللاتي', 'اللواتي', 'اللذان', 'اللتان',
    'من', 'ما', 'مَن', 'مَا',
    # Personal pronouns
    'أنا', 'نحن', 'أنت', 'أنتِ', 'أنتم', 'أنتما', 'أنتن',
    'هو', 'هي', 'هما', 'هم', 'هن',
    # Suffixed possessive/object pronouns (in WordPiece form)
    'ها', 'هم', 'هن', 'كم', 'كن', 'نا', 'ني', 'ي', 'ك',
    # Common verbs that act like function words
    'كان', 'كانت', 'كانوا', 'يكون', 'تكون', 'يكونون',
    'صار', 'أصبح', 'أمسى', 'ظل',
    'ليس', 'ليست',  # included again in case (still kept via KEEP_NEGATION)
    # Existence / particles
    'إن', 'أن', 'إنّ', 'أنّ', 'كأن', 'لأن', 'لكي', 'كي',
    'قد', 'قط', 'لقد', 'إذا', 'إذ', 'إذن', 'لو', 'لولا',
    'أيضا', 'أيضاً', 'كذلك', 'كما', 'هكذا',
    # Quantifiers / determiners
    'كل', 'بعض', 'جميع', 'معظم', 'أكثر', 'أقل',
    'نفس', 'ذات', 'ذو', 'ذي', 'صاحب',
    # Question particles
    'هل', 'كيف', 'متى', 'أين', 'لماذا', 'كم', 'أي', 'أيها',
    # Common adverbs
    'جدا', 'جداً', 'فقط', 'أيضا', 'دائما', 'دائماً', 'أبدا', 'أبداً',
    'حيث', 'بينما', 'عندما', 'حينما', 'حالما',
    # Connectors
    'وإذا', 'فإذا', 'وكذلك', 'فكذلك', 'وأيضا', 'وفي', 'ومن', 'وإلى',
    # Latin connectives sometimes appear
    'a', 'an', 'the', 'of', 'to', 'in', 'on', 'at', 'is', 'are',
    'and', 'or', 'but', 'so', 'as', 'by', 'for', 'with', 'from', 'this',
    'that', 'these', 'those', 'it', 'its',
}

# Remove negations from the stopword set (we want to keep them)
ARABIC_STOPWORDS = ARABIC_STOPWORDS - KEEP_NEGATION

def is_content_word(token, min_length=2):
    # Returns True if the token should be DISPLAYED as a meaningful content
    # word; False if it's a stopword / punctuation / special / fragment.

    if not isinstance(token, str) or not token:
        return False

    # 1. Special tokens
    if token in SPECIAL_TOKENS:
        return False

    # Strip WordPiece prefix for the rest of the checks
    core = strip_wp_prefix(token)
    if not core:
        return False

    # 2. Pure punctuation
    if all(c in PUNCT for c in core):
        return False

    # 3. Pure digits
    if all(c in DIGITS for c in core):
        return False

    # 4. Single character (most are WordPiece fragments)
    if len(core) < min_length:
        return False

    # 5. Arabic stopwords (we already removed negations from this set)
    if core in ARABIC_STOPWORDS:
        return False

    # 6. WordPiece pieces that strip to a stopword
    if token.startswith('##') and core in ARABIC_STOPWORDS:
        return False

    return True

# Smoke test
test_cases = [
    ('الرواية', True),   # 'novel' — content
    ('و',       False),  # conjunction
    ('في',      False),  # preposition
    ('[SEP]',   False),  # special
    ('.',       False),  # punctuation
    ('123',     False),  # digit
    ('ا',       False),  # single letter
    ('##ها',    False),  # suffix
    ('لا',      True),   # negation — KEPT
    ('ليس',     True),   # negation — KEPT
    ('السلطان', True),   # 'sultan' — content
    ('hello',   True),   # not a stopword, len > 1
    ('the',     False),  # English stopword
    ('كتاب',    True),   # 'book' — content
]
print('Content-word filter smoke test:')
all_pass = True
for tok, expected in test_cases:
    actual = is_content_word(tok)
    mark = '✅' if actual == expected else '❌'
    if actual != expected:
        all_pass = False
    print(f'  {mark}  {tok!r:>15}  expected={expected}  got={actual}')
print('All pass:', all_pass)

Content-word filter smoke test:
  ✅        'الرواية'  expected=True  got=True
  ✅              'و'  expected=False  got=False
  ✅             'في'  expected=False  got=False
  ✅          '[SEP]'  expected=False  got=False
  ✅              '.'  expected=False  got=False
  ✅            '123'  expected=False  got=False
  ✅              'ا'  expected=False  got=False
  ✅           '##ها'  expected=False  got=False
  ✅             'لا'  expected=True  got=True
  ✅            'ليس'  expected=True  got=True
  ✅        'السلطان'  expected=True  got=True
  ✅          'hello'  expected=True  got=True
  ✅            'the'  expected=False  got=False
  ✅           'كتاب'  expected=True  got=True
All pass: True


## 5. Load corpus, tokenizer, model

You only need cells 6-8 if you haven't run v2 yet (or if you deleted the
CSVs). If `attributions_attention.csv`, `attributions_lime.csv`, and
`attributions_shap.csv` already exist in `XAI_DIR`, **skip ahead to cell 9**.

In [ ]:
path = CONFIG['corpus_path']
df = pd.read_csv(path) if path.endswith('.csv') else pd.read_parquet(path)
test_df = df[df.split == 'test'].copy().reset_index(drop=True)
print(f'Test set: {len(test_df)} samples')

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

LABEL2ID = {'human': 0, 'ai': 1}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

class AeroCamelBertX(nn.Module):
    def __init__(self, model_name, num_labels=2, dropout=0.3, freeze_encoder=True):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name, output_attentions=True)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, num_labels),
        )
        self.frozen = freeze_encoder
        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False
            self.encoder.eval()
    def train(self, mode=True):
        super().train(mode)
        if self.frozen:
            self.encoder.eval()
        return self
    def forward(self, input_ids, attention_mask, return_attentions=False):
        with torch.set_grad_enabled(not self.frozen):
            out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        logits = self.classifier(cls)
        if return_attentions:
            return logits, out.attentions
        return logits

model = AeroCamelBertX(CONFIG['model_name'], num_labels=CONFIG['num_labels'],
                      dropout=CONFIG['dropout'], freeze_encoder=True).to(device)
ckpt = torch.load(f'{CONFIG["checkpoint_dir"]}/aero_baseline_best.pt',
                  map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded checkpoint (val F1 {ckpt["val_f1"]:.4f}, epoch {ckpt["epoch"]})')

@torch.no_grad()
def predict_proba(texts, batch_size=16, max_length=None):
    if isinstance(texts, str):
        texts = [texts]
    if max_length is None:
        max_length = CONFIG['max_length']
    probs_all = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(list(batch), truncation=True, padding=True,
                        max_length=max_length, return_tensors='pt').to(device)
        logits = model(enc['input_ids'], enc['attention_mask'])
        probs = F.softmax(logits, dim=-1).cpu().numpy()
        probs_all.append(probs)
    return np.concatenate(probs_all, axis=0)

Test set: 1262 samples


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-msa
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded checkpoint (val F1 0.9286, epoch 10)


## 6. Attention (skip if CSV exists)

In [ ]:
ATTN_CSV = f'{XAI_DIR}/attributions_attention.csv'

if os.path.exists(ATTN_CSV) and len(pd.read_csv(ATTN_CSV)) >= len(test_df):
    print(f'✅ Attention CSV already complete ({ATTN_CSV}) — skipping.')
else:
    @torch.no_grad()
    def compute_attention(text, max_length=None):
        if max_length is None: max_length = CONFIG['max_length']
        enc = tokenizer(text, truncation=True, padding=False,
                        max_length=max_length, return_tensors='pt').to(device)
        logits, attentions = model(enc['input_ids'], enc['attention_mask'],
                                    return_attentions=True)
        last = attentions[-1][0]
        cls_to_token = last[:, 0, :].mean(dim=0)
        tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
        pairs = []
        for tok, score in zip(tokens, cls_to_token.cpu().tolist()):
            if tok == tokenizer.pad_token: break
            pairs.append((tok, float(score)))
        pred = int(logits.argmax(-1).item())
        prob_ai = float(F.softmax(logits, dim=-1)[0, 1].item())
        return {'tokens_scores': pairs, 'pred': pred, 'prob_ai': prob_ai}

    done = set(pd.read_csv(ATTN_CSV)['sample_idx'].tolist()) if os.path.exists(ATTN_CSV) else set()
    print(f'Resuming attention from {len(done)}')
    rows = []; t0 = time.time(); flush = 50
    header_written = os.path.exists(ATTN_CSV)
    for i, row in test_df.iterrows():
        if i in done: continue
        r = compute_attention(row['text'])
        rows.append({
            'sample_idx': i, 'label': row['label'], 'domain': row['domain'],
            'llm': row['llm'], 'pred': r['pred'], 'prob_ai': r['prob_ai'],
            'tokens_scores': json.dumps(r['tokens_scores'], ensure_ascii=False),
        })
        if len(rows) >= flush:
            pd.DataFrame(rows).to_csv(ATTN_CSV, mode='a',
                                      header=not header_written, index=False)
            header_written = True; rows = []
            print(f'  {i+1}/{len(test_df)}')
    if rows:
        pd.DataFrame(rows).to_csv(ATTN_CSV, mode='a',
                                  header=not header_written, index=False)
    print(f'✅ Attention done in {(time.time()-t0)/60:.1f} min')

✅ Attention CSV already complete (/content/drive/MyDrive/aero_camelbert_x/logs/xai/attributions_attention.csv) — skipping.


## 7. LIME (skip if CSV exists)

In [ ]:
LIME_CSV = f'{XAI_DIR}/attributions_lime.csv'

if os.path.exists(LIME_CSV) and len(pd.read_csv(LIME_CSV)) >= len(test_df):
    print(f'✅ LIME CSV already complete — skipping.')
else:
    from lime.lime_text import LimeTextExplainer
    LIME_NUM_SAMPLES = 500; LIME_NUM_FEATURES = 30
    explainer = LimeTextExplainer(class_names=['human', 'ai'],
                                  random_state=CONFIG['seed'])
    done = set(pd.read_csv(LIME_CSV)['sample_idx'].tolist()) if os.path.exists(LIME_CSV) else set()
    print(f'Resuming LIME from {len(done)}')
    rows = []; t0 = time.time(); flush = 10
    header_written = os.path.exists(LIME_CSV)
    for i, row in test_df.iterrows():
        if i in done: continue
        try:
            exp = explainer.explain_instance(row['text'], predict_proba,
                num_features=LIME_NUM_FEATURES, num_samples=LIME_NUM_SAMPLES,
                labels=(1,))
            token_weights = exp.as_list(label=1)
            probs = predict_proba([row['text']])[0]
            rows.append({
                'sample_idx': i, 'label': row['label'], 'domain': row['domain'],
                'llm': row['llm'], 'pred': int(probs.argmax()),
                'prob_ai': float(probs[1]),
                'tokens_scores': json.dumps(token_weights, ensure_ascii=False),
            })
        except Exception as e:
            print(f'  ⚠️ LIME failed on {i}: {e}')
        if len(rows) >= flush:
            pd.DataFrame(rows).to_csv(LIME_CSV, mode='a',
                                      header=not header_written, index=False)
            header_written = True; rows = []
            n_done = i + 1 - len(done)
            rate = n_done / max(time.time()-t0, 1e-3)
            remain = (len(test_df) - i - 1) / max(rate, 1e-9) / 60
            print(f'  LIME {i+1}/{len(test_df)} (~{remain:.0f} min left)')
    if rows:
        pd.DataFrame(rows).to_csv(LIME_CSV, mode='a',
                                  header=not header_written, index=False)
    print(f'✅ LIME done in {(time.time()-t0)/60:.1f} min')

✅ LIME CSV already complete — skipping.


## 8. SHAP (skip if CSV exists)

In [ ]:
SHAP_CSV = f'{XAI_DIR}/attributions_shap.csv'

if os.path.exists(SHAP_CSV) and len(pd.read_csv(SHAP_CSV)) >= len(test_df):
    print(f'✅ SHAP CSV already complete — skipping.')
else:
    import shap
    SHAP_MIN_EVALS = 400; SHAP_MAX_EVALS = 1500; NEAR_ZERO_THRESHOLD = 1e-5
    masker = shap.maskers.Text(tokenizer)
    def predict_proba_for_shap(texts):
        return predict_proba(list(texts))
    shap_explainer = shap.Explainer(predict_proba_for_shap, masker,
                                    output_names=['human', 'ai'])
    def estimate_evals(text):
        n = len(tokenizer.tokenize(text))
        target = 2 * n + 20
        return max(SHAP_MIN_EVALS, min(SHAP_MAX_EVALS, target))

    done = set(pd.read_csv(SHAP_CSV)['sample_idx'].tolist()) if os.path.exists(SHAP_CSV) else set()
    print(f'Resuming SHAP from {len(done)}')
    rows = []; t0 = time.time(); flush = 10
    header_written = os.path.exists(SHAP_CSV)
    for i, row in test_df.iterrows():
        if i in done: continue
        try:
            n_evals = estimate_evals(row['text'])
            sv = shap_explainer([row['text']], max_evals=n_evals,
                                batch_size=16, silent=True)
            tokens = list(sv.data[0])
            ai_scores = sv.values[0, :, 1].tolist()
            token_weights = [(t, s) for t, s in zip(tokens, ai_scores)
                             if abs(s) >= NEAR_ZERO_THRESHOLD]
            probs = predict_proba([row['text']])[0]
            rows.append({
                'sample_idx': i, 'label': row['label'], 'domain': row['domain'],
                'llm': row['llm'], 'pred': int(probs.argmax()),
                'prob_ai': float(probs[1]),
                'n_tokens': len(tokens), 'n_evals': n_evals,
                'n_kept': len(token_weights),
                'tokens_scores': json.dumps(token_weights, ensure_ascii=False),
            })
        except Exception as e:
            print(f'  ⚠️ SHAP failed on {i}: {e}')
        if len(rows) >= flush:
            pd.DataFrame(rows).to_csv(SHAP_CSV, mode='a',
                                      header=not header_written, index=False)
            header_written = True; rows = []
            n_done = i + 1 - len(done)
            rate = n_done / max(time.time()-t0, 1e-3)
            remain = (len(test_df) - i - 1) / max(rate, 1e-9) / 60
            print(f'  SHAP {i+1}/{len(test_df)} (~{remain:.0f} min left)')
    if rows:
        pd.DataFrame(rows).to_csv(SHAP_CSV, mode='a',
                                  header=not header_written, index=False)
    print(f'✅ SHAP done in {(time.time()-t0)/60:.1f} min')

✅ SHAP CSV already complete — skipping.


## 9. Visualizations — content tokens only

Applies `is_content_word()` before sorting top-15 or rendering heatmaps.

In [ ]:
attn_df = pd.read_csv(f'{XAI_DIR}/attributions_attention.csv')
lime_df = pd.read_csv(f'{XAI_DIR}/attributions_lime.csv')
shap_df = pd.read_csv(f'{XAI_DIR}/attributions_shap.csv')

attn_df['correct'] = attn_df.apply(
    lambda r: (r['label']=='human' and r['pred']==0) or
              (r['label']=='ai'    and r['pred']==1), axis=1)

pick_idx = []
for label, correct in [('human', True), ('ai', True),
                       ('human', False), ('ai', False)]:
    sub = attn_df[(attn_df['label']==label) & (attn_df['correct']==correct)]
    pick_idx += sub['sample_idx'].head(3).tolist()
print(f'Selected {len(pick_idx)} samples')

Selected 12 samples


In [ ]:
def filter_content(tokens, scores):
    # Return only the (token, score) pairs whose token is a content word.
    filtered = [(t, s) for t, s in zip(tokens, scores) if is_content_word(t)]
    if not filtered:
        return tokens, scores  # fall back to unfiltered if everything got nuked
    toks = [t for t, _ in filtered]
    sc   = [s for _, s in filtered]
    return toks, sc

def normalize_scores(scores):
    s = np.array(scores, dtype=float)
    m = np.abs(s).max()
    return s / m if m > 1e-9 else s

def render_heatmap_strip(tokens, scores, title, ax):
    # Filter, then render right-to-left for Arabic reading order
    tokens, scores = filter_content(tokens, scores)
    # Top-40 by |score| if there are still too many to show
    if len(tokens) > 40:
        order = np.argsort(np.abs(scores))[::-1][:40]
        order = sorted(order.tolist())
        tokens = [tokens[k] for k in order]
        scores = [scores[k] for k in order]
    s = normalize_scores(scores)
    n = len(tokens)
    cmap = plt.cm.RdBu_r
    for k, (t, val) in enumerate(zip(tokens, s)):
        x = (n - 1 - k)   # RTL
        disp = reshape_ar(t)
        ax.text(x, 0, disp, ha='center', va='center', fontsize=10,fontproperties=AR_FONT_PROP,
                bbox=dict(facecolor=cmap((val+1)/2), edgecolor='none',
                          boxstyle='round,pad=0.3'))
    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(-0.5, 0.5)
    ax.axis('off')
    ax.set_title(title + '  (content words only)', fontsize=11, loc='right')

def render_topk_bar(tokens, scores, k, title, ax):
    # Filter first, then rank and plot
    tokens, scores = filter_content(tokens, scores)
    s = np.array(scores, dtype=float)
    nonzero = np.where(np.abs(s) >= 1e-6)[0]
    if len(nonzero) == 0:
        ax.text(0.5, 0.5, '(no non-zero content-word attributions)',
                ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')
        ax.set_title(title + '  (content)', fontsize=11, loc='left')
        return
    s_nz = s[nonzero]
    t_nz = [tokens[j] for j in nonzero]
    order = np.argsort(np.abs(s_nz))[::-1][:k]
    t_sel = [reshape_ar(t_nz[j]) for j in order]
    s_sel = s_nz[order]
    colors = ['#B23A48' if v >= 0 else '#1F4068' for v in s_sel]
    y = np.arange(len(t_sel))[::-1]
    ax.barh(y, s_sel, color=colors)
    ax.set_yticks(y)
    ax.set_yticklabels(t_sel, fontsize=11, fontproperties=AR_FONT_PROP)
    ax.axvline(0, color='gray', linewidth=0.6)
    ax.set_title(title + '  (content top-' + str(k) + ')', fontsize=11, loc='left')
    ax.set_xlabel('attribution score')

def parse_ts(json_str):
    pairs = json.loads(json_str)
    toks  = [p[0] for p in pairs]
    sc    = [p[1] for p in pairs]
    return toks, sc

def make_figure(sample_idx):
    a = attn_df[attn_df.sample_idx == sample_idx].iloc[0]
    l = lime_df[lime_df.sample_idx == sample_idx].iloc[0]
    s = shap_df[shap_df.sample_idx == sample_idx].iloc[0]

    tok_a, sc_a = parse_ts(a['tokens_scores'])
    tok_l, sc_l = parse_ts(l['tokens_scores'])
    tok_s, sc_s = parse_ts(s['tokens_scores'])

    title = (f"sample {sample_idx} | true={a['label']} | "
             f"pred={'ai' if a['pred']==1 else 'human'} | "
             f"P(ai)={a['prob_ai']:.3f} | "
             f"domain={a['domain']} | llm={a['llm']}")

    # Heatmaps
    n_tokens = min(40, max(len([t for t in tok_a if is_content_word(t)]), 8))
    fig_w = min(20, max(8, n_tokens*0.45))
    fig, axes = plt.subplots(3, 1, figsize=(fig_w, 4.5))
    fig.suptitle(title, fontsize=10)
    render_heatmap_strip(tok_a, sc_a, 'Attention',  axes[0])
    render_heatmap_strip(tok_l, sc_l, 'LIME',       axes[1])
    render_heatmap_strip(tok_s, sc_s, 'SHAP',       axes[2])
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(f'{FIG_DIR}/sample_{sample_idx:04d}_heatmap_content.png',
                dpi=150, bbox_inches='tight')
    plt.close(fig)

    # Bar charts
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(title, fontsize=10)
    render_topk_bar(tok_a, sc_a, 15, 'Attention', axes[0])
    render_topk_bar(tok_l, sc_l, 15, 'LIME',      axes[1])
    render_topk_bar(tok_s, sc_s, 15, 'SHAP',      axes[2])
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig(f'{FIG_DIR}/sample_{sample_idx:04d}_topk_content.png',
                dpi=150, bbox_inches='tight')
    plt.close(fig)

for sidx in pick_idx:
    try:
        make_figure(sidx)
    except Exception as e:
        print(f'  ⚠️ figure failed for sample {sidx}: {e}')

print(f'✅ Content-only figures written to {FIG_DIR}')
print(f'   Look for filenames ending in _heatmap_content.png and _topk_content.png')

/tmp/ipykernel_1467/2753611134.py:91: UserWarning: Glyph 65010 (\N{ARABIC LIGATURE ALLAH ISOLATED FORM}) missing from font(s) DejaVu Sans.
  fig.tight_layout(rect=[0, 0, 1, 0.96])
/tmp/ipykernel_1467/2753611134.py:92: UserWarning: Glyph 65010 (\N{ARABIC LIGATURE ALLAH ISOLATED FORM}) missing from font(s) DejaVu Sans.
  fig.savefig(f'{FIG_DIR}/sample_{sample_idx:04d}_heatmap_content.png',
/tmp/ipykernel_1467/2753611134.py:91: UserWarning: Glyph 65010 (\N{ARABIC LIGATURE ALLAH ISOLATED FORM}) missing from font(s) DejaVu Sans.
  fig.tight_layout(rect=[0, 0, 1, 0.96])
/tmp/ipykernel_1467/2753611134.py:92: UserWarning: Glyph 65010 (\N{ARABIC LIGATURE ALLAH ISOLATED FORM}) missing from font(s) DejaVu Sans.
  fig.savefig(f'{FIG_DIR}/sample_{sample_idx:04d}_heatmap_content.png',


✅ Content-only figures written to /content/drive/MyDrive/aero_camelbert_x/logs/xai/figures
   Look for filenames ending in _heatmap_content.png and _topk_content.png


## 10. Global feature importance — content words only

In [ ]:
def global_top_content_tokens(att_csv, k=50, min_occurrences=5):
    df = pd.read_csv(att_csv)
    sums   = defaultdict(float)
    counts = defaultdict(int)
    for _, r in df.iterrows():
        pairs = json.loads(r['tokens_scores'])
        for tok, score in pairs:
            if not is_content_word(tok):
                continue                 # the v3 filter
            sums[tok]   += abs(float(score))
            counts[tok] += 1
    out = []
    for tok, total in sums.items():
        if counts[tok] < min_occurrences:
            continue
        out.append({
            'token'         : tok,
            'token_display' : reshape_ar(tok),
            'mean_abs_score': total / counts[tok],
            'n_occurrences' : counts[tok],
        })
    return pd.DataFrame(out).sort_values('mean_abs_score', ascending=False).head(k)

g_attn = global_top_content_tokens(f'{XAI_DIR}/attributions_attention.csv').assign(method='attention')
g_lime = global_top_content_tokens(f'{XAI_DIR}/attributions_lime.csv').assign(method='lime')
g_shap = global_top_content_tokens(f'{XAI_DIR}/attributions_shap.csv').assign(method='shap')

g_all = pd.concat([g_attn, g_lime, g_shap], ignore_index=True)
g_all.to_csv(f'{XAI_DIR}/global_top_content_tokens.csv', index=False)

print('Top-20 CONTENT tokens (LIME):')
print(g_lime.head(20)[['token_display', 'mean_abs_score', 'n_occurrences']]
      .to_string(index=False))
print()
print('Top-20 CONTENT tokens (SHAP):')
print(g_shap.head(20)[['token_display', 'mean_abs_score', 'n_occurrences']]
      .to_string(index=False))

Top-20 CONTENT tokens (LIME):
token_display  mean_abs_score  n_occurrences
      ﻲﻘﻠﺘﻤﻟﺍ        0.074205              6
      ﺩﺎﺼﺘﻗﻻﺍ        0.074198              8
        ﻪﻴﻠﻋﻭ        0.070617             12
       ﺎﻫﺩﺎﻔﻣ        0.070023              6
      ﻪﻴﺣﺎﺘﻔﻣ        0.058350              6
         ﺎﻤﻟﻭ        0.058316              5
        ﺖﺼﻠﺧﻭ        0.054028              5
        ﺶﻗﺎﻨﻳ        0.053435             11
         ﺏﺩﻻﺍ        0.053411              9
      ﺕﺎﻤﻠﻜﻟﺍ        0.052658             30
         ﺍﺫﺎﻣ        0.047942              5
       ﺏﺮﻐﻤﻟﺍ        0.043745              6
         ﻲﺻﻮﺗ        0.043042              9
         ﻒﻴﻌﺿ        0.043025              6
        ﺺﺼﻘﻟﺍ        0.042619              6
      ﻪﻘﻄﻨﻤﻟﺍ        0.042571             10
          ﻊﻣﻭ        0.042288             30
         ﻝﺎﻳﺭ        0.041906              5
    ﻪﻴﺣﺎﺘﻔﻤﻟﺍ        0.041365             16
      ﺮﺋﺍﺰﺠﻟﺍ        0.040535             45

Top-20 CONTENT tokens (S

## 11. Filter audit

Confirm the filter is doing its job — what fraction of each sample's
tokens survive the content filter?

In [ ]:
def audit_filter(csv_path, method_name):
    df = pd.read_csv(csv_path)
    surviving = []
    for _, r in df.iterrows():
        pairs = json.loads(r['tokens_scores'])
        n_all = len(pairs)
        n_content = sum(1 for t, _ in pairs if is_content_word(t))
        surviving.append({
            'sample_idx': r['sample_idx'],
            'total'     : n_all,
            'content'   : n_content,
            'ratio'     : n_content / max(n_all, 1),
        })
    sdf = pd.DataFrame(surviving)
    print(f'\n{method_name}:')
    print(f'  median content tokens / sample: {sdf["content"].median():.0f}')
    print(f'  median ratio kept             : {sdf["ratio"].median()*100:.1f}%')
    print(f'  samples with <5 content toks  : {(sdf["content"] < 5).sum()}')
    return sdf

audit_filter(f'{XAI_DIR}/attributions_attention.csv', 'Attention')
audit_filter(f'{XAI_DIR}/attributions_lime.csv',      'LIME')
audit_filter(f'{XAI_DIR}/attributions_shap.csv',      'SHAP')


Attention:
  median content tokens / sample: 189
  median ratio kept             : 68.3%
  samples with <5 content toks  : 0

LIME:
  median content tokens / sample: 27
  median ratio kept             : 90.0%
  samples with <5 content toks  : 1

SHAP:
  median content tokens / sample: 256
  median ratio kept             : 94.9%
  samples with <5 content toks  : 0


,sample_idx,total,content,ratio
0,0,239,224,0.937238
1,1,279,255,0.913978
2,2,492,457,0.928862
3,3,541,507,0.937153
4,4,123,119,0.967480
...,...,...,...,...
1257,1257,194,179,0.922680
1258,1258,209,201,0.961722
1259,1259,281,271,0.964413
1260,1260,205,201,0.980488


## 12. Method agreement on top *content* token

Re-compute method agreement using only content words.

In [ ]:
def top_content_token(json_str):
    pairs = json.loads(json_str)
    content_pairs = [(t, s) for t, s in pairs if is_content_word(t)]
    if not content_pairs:
        return None
    return max(content_pairs, key=lambda p: abs(float(p[1])))[0]

attn_df['top_content'] = attn_df['tokens_scores'].apply(top_content_token)
lime_df['top_content'] = lime_df['tokens_scores'].apply(top_content_token)
shap_df['top_content'] = shap_df['tokens_scores'].apply(top_content_token)

merged = (attn_df[['sample_idx','top_content']]
          .merge(lime_df[['sample_idx','top_content']], on='sample_idx',
                 suffixes=('_attn','_lime'))
          .merge(shap_df[['sample_idx','top_content']], on='sample_idx'))
merged = merged.rename(columns={'top_content': 'top_content_shap'})

# Drop rows where any method had no content word at all (rare)
merged = merged.dropna()

merged['attn_lime_match'] = (merged['top_content_attn']==merged['top_content_lime']).astype(int)
merged['lime_shap_match'] = (merged['top_content_lime']==merged['top_content_shap']).astype(int)
merged['attn_shap_match'] = (merged['top_content_attn']==merged['top_content_shap']).astype(int)
merged['all_three_match'] = ((merged.attn_lime_match==1)&(merged.lime_shap_match==1)).astype(int)

print(f'Method agreement on top CONTENT token (n={len(merged)}):')
print(f'  Attention ↔ LIME : {merged.attn_lime_match.mean():.3%}')
print(f'  LIME      ↔ SHAP : {merged.lime_shap_match.mean():.3%}')
print(f'  Attention ↔ SHAP : {merged.attn_shap_match.mean():.3%}')
print(f'  All three agree  : {merged.all_three_match.mean():.3%}')

merged.to_csv(f'{XAI_DIR}/method_agreement_content.csv', index=False)

Method agreement on top CONTENT token (n=1262):
  Attention ↔ LIME : 4.675%
  LIME      ↔ SHAP : 0.713%
  Attention ↔ SHAP : 1.506%
  All three agree  : 0.475%


## 13. Wrap up

Phase 3 v3 complete. What's new vs v2:

| Output | v2 | v3 |
|---|---|---|
| Heatmap PNGs | `..._heatmap.png` | `..._heatmap_content.png` |
| Bar-chart PNGs | `..._topk.png` | `..._topk_content.png` |
| Global tokens CSV | `global_top_tokens.csv` | `global_top_content_tokens.csv` |
| Method agreement CSV | `method_agreement.csv` | `method_agreement_content.csv` |
| Raw attribution CSVs | unchanged | unchanged ← same files, no rerun needed |

**The raw CSVs are untouched** — if a reviewer later asks "what about
the stopwords?", you can re-render with the filter disabled by editing
`filter_content` to return its inputs unchanged.

**Next: Phase 4** — the dual-loop continual learning gate.

In [ ]:
print('=== Phase 3 v3 summary ===')
print(f'XAI dir      : {XAI_DIR}')
print(f'Negation kept: لا، ليس، لم، ما، لن، غير، بدون')
print(f'Stopwords    : ~250 (CAMeL + Khoja + NLTK Arabic, minus negations)')
print(f'New figures  : {sum(1 for f in os.listdir(FIG_DIR) if "content" in f)} content-filtered PNGs')
print()
print('Ready for Phase 4 ✅')

=== Phase 3 v3 summary ===
XAI dir      : /content/drive/MyDrive/aero_camelbert_x/logs/xai
Negation kept: لا، ليس، لم، ما، لن، غير، بدون
Stopwords    : ~250 (CAMeL + Khoja + NLTK Arabic, minus negations)
New figures  : 24 content-filtered PNGs

Ready for Phase 4 ✅
